# Pasamos los datos a una base de datos SQL

In [1]:
%pip install mysql-connector-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import mysql.connector
import pandas as pd

In [3]:
import getpass  

password = getpass.getpass()

In [4]:
conn = mysql.connector.connect(host='localhost',
                        user='root',
                        passwd=password)

In [5]:
cursor = conn.cursor()

cursor

# Creamos la carpeta de Logística

In [6]:
cursor.execute("CREATE DATABASE IF NOT EXISTS LOGISTICA")

# Creamos la tabla Cliente

In [7]:
cursor.execute("USE LOGISTICA")

In [8]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS cliente (
        id_cliente VARCHAR(50) PRIMARY KEY,
        nombre_cliente VARCHAR(255) NOT NULL,
        tipo_cliente VARCHAR(255) NOT NULL,
        dias_plazo_credito INT NOT NULL,
        tipo_carga_principal VARCHAR(255) NOT NULL,
        estado_cuenta VARCHAR(255) NOT NULL,
        fecha_inicio_contrato DATETIME NOT NULL,
        potencial_ingresos_anuales DECIMAL NOT NULL,
        ingreso_eur DECIMAL NOT NULL
    )
""")
print("Table cliente created successfully!")

Table cliente created successfully!


In [9]:
df_sql1= pd.read_csv("df_clientes_estudio.csv")

In [10]:
tupla1=list(df_sql1.itertuples(index=False,name=None))

In [11]:
print(len(tupla1[0]))

9


In [12]:
cursor.executemany("INSERT INTO cliente (id_cliente, nombre_cliente, tipo_cliente, dias_plazo_credito, tipo_carga_principal, estado_cuenta, fecha_inicio_contrato, potencial_ingresos_anuales, ingreso_eur) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)", tupla1)
conn.commit()
print(f"{cursor.rowcount} items inserted!")

200 items inserted!


# Creamos la tabla rutas

In [13]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS rutas (
        id_ruta VARCHAR(50) PRIMARY KEY,
        ciudad_origen VARCHAR(255) NOT NULL,
        estado_origen VARCHAR(255) NOT NULL,
        ciudad_destino VARCHAR(255) NOT NULL,
        estado_destino VARCHAR(255) NOT NULL,
        distancia_tipica_millas DECIMAL NOT NULL,
        distancia_km DECIMAL NOT NULL,
        tarifa_base_por_milla DECIMAL NOT NULL,
        tarifa_base_milla_eur DECIMAL NOT NULL,
        tasa_de_recargo_combustible DECIMAL NOT NULL,
        dias_de_transito_tipicos DECIMAL NOT NULL
    )
""")
print("Table rutas created successfully!")

Table rutas created successfully!


In [14]:
df_sql2= pd.read_csv("df_rutas_estudio.csv")

In [15]:
tupla2=list(df_sql2.itertuples(index=False,name=None))


In [16]:
print(len(tupla2[0]))

11


In [17]:

cursor.executemany("INSERT INTO rutas (id_ruta, ciudad_origen, estado_origen, ciudad_destino, estado_destino, distancia_tipica_millas, distancia_km, tarifa_base_por_milla, tarifa_base_milla_eur, tasa_de_recargo_combustible, dias_de_transito_tipicos) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)", tupla2)
conn.commit()
print(f"{cursor.rowcount} items inserted!")

58 items inserted!


# Creamos la tabla de Cargas

In [18]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS cargas (
        id_carga VARCHAR(50) PRIMARY KEY,
        id_cliente VARCHAR(50),
        id_ruta VARCHAR(50),
        fecha_carga DATETIME NOT NULL,
        tipo_carga VARCHAR(255) NOT NULL,
        peso_lbs DECIMAL NOT NULL,
        peso_kg DECIMAL NOT NULL,
        piezas DECIMAL NOT NULL,
        ingreso_dolar DECIMAL NOT NULL,
        ingreso_eur DECIMAL NOT NULL,
        recargo_combustible DECIMAL NOT NULL,
        cargos_accesorios DECIMAL NOT NULL,
        estado_carga VARCHAR(255) NOT NULL,
        tipo_reserva VARCHAR(255) NOT NULL,
        
        FOREIGN KEY (id_cliente) REFERENCES cliente(id_cliente),
        FOREIGN KEY (id_ruta) REFERENCES rutas(id_ruta)
    )
""")
print("Table cargas created successfully!")


Table cargas created successfully!


In [19]:
df_sql3= pd.read_csv("df_carga_estudio.csv")

In [20]:
tupla3=list(df_sql3.itertuples(index=False,name=None))

In [21]:
print(len(tupla3[0]))

14


In [22]:
cursor.executemany("INSERT INTO cargas (id_carga , id_cliente , id_ruta , fecha_carga , tipo_carga , peso_lbs , peso_kg , piezas , ingreso_dolar , ingreso_eur , recargo_combustible , cargos_accesorios , estado_carga , tipo_reserva) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)", tupla3)
conn.commit()
print(f"{cursor.rowcount} items inserted!")

85410 items inserted!


# Creamos la tabla Viajes

In [23]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS viajes (
        id_viaje VARCHAR(50) PRIMARY KEY,
        id_carga VARCHAR(50),
        fecha_despacho DATETIME NOT NULL,
        distancia_real_millas DECIMAL NOT NULL,
        distancia_km DECIMAL NOT NULL,
        duracion_real_horas DECIMAL NOT NULL,
        galones_combustible_usados DECIMAL NOT NULL,
        mpg_promedio DECIMAL NOT NULL,
        horas_de_ralenti_RPM DECIMAL NOT NULL,
        estado_viaje VARCHAR(255) NOT NULL,
    
        FOREIGN KEY (id_carga) REFERENCES cargas(id_carga)
    )
""")
print("Table viajes created successfully!")


Table viajes created successfully!


In [24]:
df_sql= pd.read_csv("df_viajes_estudio.csv")

In [25]:
tupla=list(df_sql.itertuples(index=False,name=None))


In [26]:
print(len(tupla[0]))

10


In [27]:

cursor.executemany("INSERT INTO viajes (id_viaje , id_carga , fecha_despacho, distancia_real_millas, distancia_km, duracion_real_horas, galones_combustible_usados, mpg_promedio, horas_de_ralenti_RPM, estado_viaje) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)", tupla)
conn.commit()
print(f"{cursor.rowcount} items inserted!")

85410 items inserted!


# Creamos la tabla Compra combustible

In [28]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS compra_combustible (
    id_compra_combustible VARCHAR(50) PRIMARY KEY,
    id_viaje VARCHAR(50),
    fecha_compra DATE,
    ciudad VARCHAR(200),
    estado VARCHAR(200),
    galones DECIMAL,
    precio_por_galon DECIMAL,
    costo_total_usd DECIMAL,
    costo_total_eur DECIMAL,
    
    FOREIGN KEY (id_viaje) REFERENCES viajes(id_viaje)
    )
""")
print("Table compra_combustible created successfully!")


Table compra_combustible created successfully!


In [29]:
df_sql4= pd.read_csv("df_combustible_estudio.csv")

In [30]:
tupla4=list(df_sql4.itertuples(index=False,name=None))

In [31]:
print(len(tupla4[0]))

9


In [32]:
cursor.executemany("INSERT INTO compra_combustible (id_compra_combustible, id_viaje, fecha_compra, ciudad, estado, galones, precio_por_galon, costo_total_usd, costo_total_eur) VALUES (%s,%s,%s,%s, %s, %s, %s, %s, %s)", tupla4)
conn.commit()
print(f"{cursor.rowcount} items inserted!")

196442 items inserted!


# Creamos la tabla Eventos de Entrega y Clima


In [33]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS eventos_entrega_clima (
    id_evento VARCHAR(50) PRIMARY KEY,
    id_carga VARCHAR(50),
    id_viaje VARCHAR(50),
    tipo_evento VARCHAR(200),
    fecha_hora_programada DATETIME,
    fecha_hora_real DATETIME,
    minutos_de_detencion DECIMAL,
    indicador_tiempo_hora VARCHAR(200),
    ciudad VARCHAR(200),
    estado VARCHAR(200),
    fecha DATE,
    hora TIME,
    tem_max_C DECIMAL,
    tem_min_C DECIMAL,
    tem_C DECIMAL,
    sens_term_max_C DECIMAL,
    sens_term_min_C DECIMAL,
    sens_term_C DECIMAL,
    rocio_C DECIMAL,
    humedad_pct DECIMAL,
    precip_mm DECIMAL,
    prob_precip_pct DECIMAL,
    cob_precip_pct DECIMAL,
    tipo_precip VARCHAR(200),
    nieve_mm DECIMAL,
    prof_nieve_mm DECIMAL,
    rafaga_viento_km_h DECIMAL,
    vel_viento_km_h DECIMAL,
    dir_viento_gr DECIMAL,
    nubosidad_pct DECIMAL,
    visibilidad_km DECIMAL,
    riesgo_severo VARCHAR(200),
    condiciones VARCHAR(200),
    descripcion VARCHAR(200),
    icono VARCHAR(200),
        
    FOREIGN KEY (id_viaje) REFERENCES viajes(id_viaje),
    FOREIGN KEY (id_carga) REFERENCES cargas(id_carga)
    )
""")
print("Table eventos_entrega_clima created successfully!")


Table eventos_entrega_clima created successfully!


In [34]:
df_sql5 = pd.read_csv("df_union_estudio.csv")

In [35]:
tupla5=list(df_sql5.itertuples(index=False,name=None))

In [36]:
print(len(tupla5[0]))

35


In [38]:
cursor.executemany("INSERT INTO eventos_entrega_clima (id_evento, id_carga, id_viaje, tipo_evento, fecha_hora_programada, fecha_hora_real, minutos_de_detencion, indicador_tiempo_hora, ciudad, estado, fecha, hora, tem_max_C, tem_min_C, tem_C, sens_term_max_C, sens_term_min_C, sens_term_C, rocio_C, humedad_pct, precip_mm, prob_precip_pct, cob_precip_pct, tipo_precip, nieve_mm, prof_nieve_mm, rafaga_viento_km_h, vel_viento_km_h, dir_viento_gr, nubosidad_pct, visibilidad_km, riesgo_severo, condiciones, descripcion, icono) VALUES (%s,%s,%s,%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)", tupla5)
conn.commit()
print(f"{cursor.rowcount} items inserted!")

66146 items inserted!


# Creamos la tabla Incidentes

In [39]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS incidentes (
    id_incidente VARCHAR(50) PRIMARY KEY,
    id_viaje VARCHAR(50),
    fecha_incidente DATE,
    tipo_incidente VARCHAR(200),
    ciudad VARCHAR(200),
    estado VARCHAR(200),
    indicador_de_falla VARCHAR(200),
    indicador_de_lesion VARCHAR(200),
    costo_vehiculo DECIMAL,
    costo_carga DECIMAL,
    monto_reclamacion_dolar DECIMAL,
    monto_reclamacion_eur DECIMAL,
    indicador_prevenible VARCHAR(200),
    nivel VARCHAR(200),
    causa VARCHAR(200),
    
    FOREIGN KEY (id_viaje) REFERENCES viajes(id_viaje)
    )
""")
print("Table incidentes created successfully!")


Table incidentes created successfully!


In [40]:
df_sql6= pd.read_csv("df_incidente_estudio.csv")

In [41]:
tupla6=list(df_sql6.itertuples(index=False,name=None))

In [42]:
print(len(tupla6[0]))

15


In [43]:
cursor.executemany("INSERT INTO incidentes (id_incidente, id_viaje, fecha_incidente, tipo_incidente, ciudad, estado, indicador_de_falla, indicador_de_lesion, costo_vehiculo, costo_carga, monto_reclamacion_dolar, monto_reclamacion_eur, indicador_prevenible, nivel, causa) VALUES (%s,%s,%s,%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)", tupla6)
conn.commit()
print(f"{cursor.rowcount} items inserted!")

170 items inserted!


Una vez finalizada la carga de datos en SQL, se puede proceder a realizar las consultas SQL para obtener la información necesaria. Seguiendo con el análisis correspondiente.